# 🧪 ADK Application Testing

This notebook demonstrates how to test an ADK (Agent Development Kit) application.
It covers both local and remote testing, both with Agent Engine and Cloud Run.

> **Note**: This notebook assumes that the agent files are stored in the `app` folder. If your agent files are located in a different directory, please update all relevant file paths and references accordingly.

## Set Up Your Environment

> **Note:** For best results, use the same `.venv` created for local development with `uv` to ensure dependency compatibility and avoid environment-related issues.

In [ ]:
# Uncomment the following lines if you're not using the virtual environment created by uv
# import sys

# sys.path.append("../")
# !pip install google-cloud-aiplatform a2a-sdk --upgrade

### Import libraries

In [ ]:
import json

import requests
import vertexai

In [ ]:
import os

from dotenv import load_dotenv

# Load environment variables from the project root's .env file
load_dotenv(dotenv_path="../.env")

# Verify that the variables have loaded successfully (without displaying sensitive secrets)
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
model = os.getenv("MODEL")
REGION = os.getenv("GOOGLE_CLOUD_REGION")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION")

print(f"Loaded environment variables for project: {project_id}")
print(f"Using Model: {model}")
print(f"Region: {REGION}")
print(f"Location: {LOCATION}")



### Initialize Vertex AI Client

In [ ]:
# Initialize the Vertex AI client
client = vertexai.Client(
    location=REGION,
)

### Setup Tracing

In [ ]:
from app.app_utils.telemetry import setup_telemetry
setup_telemetry()

## If you are using Agent Engine
See more documentation at [Agent Engine Overview](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview)

### Remote Testing

In [ ]:
# Set to None to auto-detect from ./deployment_metadata.json, or specify manually
# "projects/PROJECT_ID/locations/us-central1/reasoningEngines/ENGINE_ID"
REMOTE_RUNTIME_ENGINE_ID = None

if REMOTE_RUNTIME_ENGINE_ID is None:
    try:
        with open("../deployment_metadata.json") as f:
            metadata = json.load(f)
            RUNTIME_ENGINE_ID = metadata.get("remote_agent_runtime_id")
    except (FileNotFoundError, json.JSONDecodeError):
        pass

print(f"Using REASONING_ENGINE_ID: {RUNTIME_ENGINE_ID}")
# Get the existing agent engine
remote_agent_engine = client.agent_engines.get(name=RUNTIME_ENGINE_ID)

In [ ]:
async for event in remote_agent_engine.async_stream_query(
    message="hi!", user_id="test"
):
    print(event)

### Local Testing

You can import directly the AgentEngineApp class within your environment. 
To run the agent locally, follow these steps:
1. Make sure all required packages are installed in your environment
2. The recommended approach is to use the same virtual environment created by the 'uv' tool
3. You can set up this environment by running 'make install' from your agent's root directory
4. Then select this kernel (.venv folder in your project) in your Jupyter notebook to ensure all dependencies are available

In [ ]:
# Import the actual agent_runtime from your project
from app.agent_runtime_app import agent_runtime

# Set up and initialise the local agent
agent_runtime.set_up()

In [ ]:
async for event in agent_runtime.async_stream_query(message="hi!", user_id="test"):
    print(event)
